In [1]:
# Imports
from dotenv import load_dotenv
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage, AssistantMessage, ToolMessage
from gen_ai_hub.orchestration.models.template import Template
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.config import OrchestrationConfig
from gen_ai_hub.orchestration.service import OrchestrationService
from gen_ai_hub.orchestration.models.multimodal_items import ImageItem
from IPython.display import display, Markdown, clear_output

load_dotenv()

# ChatHistory class
class ChatHistory:
    """Manages conversation history as a list of typed message objects."""

    def __init__(self):
        self._messages = []

    def append_user_message(self, content):
        """Appends a user message. Content can be a string or a list (for multimodal)."""
        self._messages.append(UserMessage(content=content))

    def append_assistant_message(self, content):
        """Appends an assistant message."""
        self._messages.append(AssistantMessage(content))

    def append_tool_message(self, content, tool_call_id):
        """Appends a tool result message."""
        self._messages.append(ToolMessage(content=content, tool_call_id=tool_call_id))

    def append_raw(self, message):
        """Appends a raw API response message (e.g., assistant message with tool_calls)."""
        self._messages.append(message)

    def get_messages(self):
        """Returns the list of messages."""
        return self._messages

    def __iter__(self):
        return iter(self._messages)

    def __repr__(self):
        lines = []
        for msg in self._messages:
            role = msg.role.value if hasattr(msg.role, 'value') else msg.role
            content = msg.content if isinstance(msg.content, str) else str(msg.content)
            lines.append(f"{role}: {content}")
        return "\n".join(lines)

    def show(self):
        """Renders the chat history as a debug view."""
        lines = []
        for msg in self._messages:
            if isinstance(msg, SystemMessage):
                lines.append(f"**system**: {msg.content}")
            elif isinstance(msg, UserMessage):
                if isinstance(msg.content, list):
                    lines.append("**user**: [image + text]")
                else:
                    lines.append(f"**user**: {msg.content}")
            elif isinstance(msg, AssistantMessage):
                lines.append(f"**assistant**: {msg.content}")
            elif isinstance(msg, ToolMessage):
                lines.append(f"**tool result** (id: {msg.tool_call_id}): {msg.content}")
            elif hasattr(msg, "tool_calls") and msg.tool_calls:
                lines.append("**tool calls**")
                for i, tc in enumerate(msg.tool_calls, 1):
                    lines.append(f"- {i}. `{tc.function.name}({tc.function.arguments})`")
            elif hasattr(msg, "content"):
                lines.append(f"**assistant**: {msg.content}")
        return Markdown("\n\n".join(lines))


# ChatClient class
class ChatClient:

    def __init__(self, system_prompt, model_name="gpt-4o", tools=None):
        """Initializes the Chat Client."""
        self._system_prompt = system_prompt
        self._model_name = model_name
        self._history = ChatHistory()
        self._tools = tools

    def _build_config(self):
        """Builds the orchestration config."""
        return OrchestrationConfig(
            template=Template(messages=[SystemMessage(self._system_prompt)], tools=self._tools),
            llm=LLM(name=self._model_name)
        )

    def _process_tool_calls(self, tool_calls):
        """Executes tool calls and adds results to history."""
        tool_map = {t.name: t for t in self._tools}
        for tc in tool_calls:
            tool = tool_map[tc.function.name]
            result = tool.execute(**tc.function.parse_arguments())
            self._history.append_tool_message(content=str(result), tool_call_id=tc.id)

    def _process_response(self, result):
        """Processes a model response, handling tool calls recursively (ReAct loop)."""
        assistant_msg = result.orchestration_result.choices[0].message

        if assistant_msg.tool_calls:
            self._history.append_raw(assistant_msg)
            self._process_tool_calls(assistant_msg.tool_calls)
            result = OrchestrationService(config=self._build_config()).run(
                history=self._history.get_messages()
            )
            return self._process_response(result)

        self._history.append_assistant_message(assistant_msg.content)
        return Markdown(assistant_msg.content)

    def get_response(self, prompt):
        """Sends a prompt to the LLM and returns the response.
        prompt can be a string or a list (for multimodal content with images)."""
        self._history.append_user_message(prompt)
        result = OrchestrationService(config=self._build_config()).run(
            history=self._history.get_messages()
        )
        return self._process_response(result)

    def get_streaming_response(self, prompt):
        """Sends a prompt to the LLM and streams the response."""
        self._history.append_user_message(prompt)
        response = OrchestrationService(config=self._build_config()).stream(
            history=self._history.get_messages()
        )
        complete_response = ""
        for chunk in response:
            delta = chunk.orchestration_result.choices[0].delta.content
            if delta:
                complete_response += delta
                clear_output(wait=True)
                display(Markdown(complete_response))
        self._history.append_assistant_message(complete_response)


# Helper for loading images
def load_image(path):
    """Load an image file and return an ImageItem for multimodal messages."""
    return ImageItem.from_file(path)

In [2]:
chat = ChatClient("Answer in a very concise and accurate way", model_name="gpt-4o")
chat.get_response("Name the planets in the solar system")

Mercury, Venus, Earth, Mars, Jupiter, Saturn, Uranus, Neptune.

In [3]:
chat.get_response("Please reverse the list")

Neptune, Uranus, Saturn, Jupiter, Mars, Earth, Venus, Mercury.